# BDB27 — Combine ceiling-binding pipeline (full run)

**Does the Combine ceiling bind on max-effort routes?** (WR primary, TE replication)

This notebook is the *runnable* deliverable: it opens, you set two things in the
**Configuration** cell below (`MODE` and — only if needed — `DATA_DIR`), then **Run All**.
No terminal required.

It drives the `src/` engine end-to-end in the frozen order

```
01_audit → 02_combine_envelope → 00_thresholds → 03_game_demand → 04_models → 05_validate
```

and renders the **≤5 writeup figures** into `outputs/figures/`.

**Works on** the ARM64 Raspberry Pi, a Windows / macOS laptop, and **Kaggle**
(auto-detects the attached dataset under `/kaggle/input/...`; otherwise uses `data/raw`).

### Reading the labels

* `sample` mode outputs are stamped **`UNVALIDATED SAMPLE OUTPUT`** — plumbing, not findings.
* `full` mode outputs are stamped **`FULL`**.
* The committed **synthetic fixture** (planted-ceiling `binding` / `null`) is stamped
  **`SYNTHETIC FIXTURE (not a finding)`** — it only proves the game-side code path runs.

Memory discipline (Pi): the notebook never loads the large tracking CSVs into pandas —
all heavy lifting is streamed inside the `src/` stages via Parquet + DuckDB.

## 1. Configuration

Edit only this cell. `MODE="sample"` runs everything that the Pi's partial data allows and
clearly flags the game-side stages as `PENDING FULL RUN`. Set `MODE="full"` once the full
`game_tracking_{2023,24,25}.csv` files are available.

In [ ]:
# ===================== CONFIG — edit me (nothing else) =====================
MODE               = "sample"   # "sample" | "full"
DATA_DIR           = None       # None = auto-detect (/kaggle/input/... else <repo>/data/raw)
FORCE              = False      # True = ignore stage checkpoints and rebuild
CLEAN              = True       # True = clear previous stage outputs for a reproducible run
RUN_SYNTHETIC_DEMO = True       # exercise the game-side path on the committed fixture
SYNTHETIC_REGIMES  = ("binding", "null")
SHOW_FIGURES       = True       # display figures inline
FIG_DPI            = 150
assert MODE in ("sample", "full"), MODE
# ==========================================================================

## 2. Bootstrap — locate repo, data, write the run config

No commands to type: this cell finds the repository, picks the data folder, writes a
scratch run-config (`bdb27_run/config.yaml`) that points the engine at the data, and prints
what will run.

In [ ]:
import importlib, io, json, sys, time, traceback, shutil
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# ---- locate the repo root (config.yaml + src/common.py) --------------------
def find_repo_root():
    seen = []
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "config.yaml").exists() and (p / "src" / "common.py").exists():
            return p
    raise RuntimeError("Cannot locate the BDB27 repo root (need config.yaml + src/common.py) "
                       "— open this notebook from inside the BDB27 checkout.")

REPO = find_repo_root()
SRC  = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))          # so `import common` etc. resolve (numeric stage names import via importlib)
import common                             # the shared engine

# bind the logger to the real stderr BEFORE any redirect, so stage logs stay visible
common.get_logger()
CFG_REPO = common.load_config(str(REPO / "config.yaml"))

# ---- pick the data directory ----------------------------------------------
def has_combine(d):
    return Path(d).is_dir() and (Path(d) / "combine_tracking.csv").exists()

def detect_data_dir(override):
    if override:
        p = Path(override).expanduser().resolve()
        if has_combine(p):
            return p
        hits = sorted(p.rglob("combine_tracking.csv"))
        return hits[0].parent if hits else p
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("combine_tracking.csv"))
        if hits:
            return hits[0].parent
    return REPO / "data" / "raw"

DATA = detect_data_dir(DATA_DIR)

# ---- scratch run-config (override only the data path; everything else stays default)
RUN_DIR = REPO / "bdb27_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CFG_PATH = RUN_DIR / "config.yaml"
run_cfg = json.loads(json.dumps(CFG_REPO))          # deep copy (config is JSON-safe)
run_cfg.pop("_config_path", None)
run_cfg["paths"]["raw"] = str(DATA)                 # absolute -> resolve() passes it through
RUN_CFG_PATH.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding="utf-8")
RUN_CFG = common.load_config(str(RUN_CFG_PATH))

OUTPUTS  = common.resolve(RUN_CFG, "outputs")
FIGDIR   = common.resolve(RUN_CFG, "figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---- which raw files are present? -----------------------------------------
RAW_FILES = ["combine_tracking.csv", "combine_results.csv", "players.csv",
             "games.csv", "player_play.csv",
             "game_tracking_2023.csv", "game_tracking_2024.csv", "game_tracking_2025.csv",
             "game_tracking_2023_sample.csv"]
present = {f: (DATA / f).exists() for f in RAW_FILES}
full_game_ok = all(present[f] for f in ("game_tracking_2023.csv", "game_tracking_2024.csv",
                                        "game_tracking_2025.csv"))
combine_ok   = present["combine_tracking.csv"] and present["combine_results.csv"]

if CLEAN:
    for d in ("audit", "envelope", "game", "models", "validate"):
        shutil.rmtree(OUTPUTS / d, ignore_errors=True)
    for f in ("thresholds.json", "run_log.csv"):
        try: (OUTPUTS / f).unlink()
        except FileNotFoundError: pass
    for f in FIGDIR.glob("*.png"):
        f.unlink()

# ---- shared output readers (used by the run + figure cells) -----------------
def out_path(*parts):
    return OUTPUTS.joinpath(*parts)

def read_csv(path, **kw):
    path = Path(path)
    return pd.read_csv(path, **kw) if path.exists() else pd.DataFrame()

def prov_of(df, fallback="UNVALIDATED SAMPLE OUTPUT"):
    if len(df) and "provenance" in df.columns:
        return str(df["provenance"].iloc[0])
    return fallback

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def is_true(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

print(f"repo root      : {REPO}")
print(f"data dir       : {DATA}")
print(f"mode           : {MODE}   (force={FORCE}, clean={CLEAN})")
print(f"Combine files  : {'present' if combine_ok else 'MISSING'}")
print(f"full game files: {'present' if full_game_ok else 'ABSENT -> game-side stages are PENDING FULL RUN'}")
print(f"run config     : {RUN_CFG_PATH.relative_to(REPO)}")
print(f"outputs        : {OUTPUTS.relative_to(REPO)}/   figures -> {FIGDIR.relative_to(REPO)}/")

## 3. Run the pipeline in order

Each stage is invoked through its own entry point (so its `--mode` / `--force` / checkpoint
behaviour and the pre-registration **freeze gate** all apply exactly as on the command line).
A stage that needs data this machine does not have is caught and reported as
`PENDING FULL RUN` rather than crashing the notebook.

In [ ]:
STAGE_ORDER = ["01_audit", "02_combine_envelope", "00_thresholds",
               "03_game_demand", "04_models", "05_validate"]
GAME_SIDE  = {"01_audit", "00_thresholds", "03_game_demand", "04_models", "05_validate"}

def run_stage(module_name, extra_args):
    """Run one src/ stage in-process via its own main(). Returns a status dict."""
    mod = importlib.import_module(module_name)          # numeric names are fine via importlib
    old = sys.argv
    sys.argv = [f"{module_name}.py", *[str(a) for a in extra_args]]
    buf = io.StringIO()
    t0, rc, err = time.time(), 0, ""
    try:
        with redirect_stdout(buf), redirect_stderr(buf):
            mod.main()
    except SystemExit as e:
        rc = 0 if e.code in (None, 0) else int(e.code)
    except BaseException as e:
        rc, err = 1, f"{type(e).__name__}: {e}"
        traceback.print_exc(file=buf)
    finally:
        sys.argv = old
    tail = [ln for ln in buf.getvalue().strip().splitlines() if ln.strip()][-6:]
    return {"stage": module_name, "ok": rc == 0, "rc": rc,
            "seconds": round(time.time() - t0, 2), "error": err, "tail": tail}

SUMMARY_FILE = {"03_game_demand": ("game", "game_demand_summary.csv"),
                "04_models":      ("models", "models_summary.csv"),
                "05_validate":    ("validate", "validation_summary.csv")}

def stage_output_status(name):
    """Honest per-stage status from the stage's OWN outputs (PENDING vs OK vs UNVALIDATED)."""
    try:
        if name == "00_thresholds":
            p = out_path("thresholds.json")
            prov = json.loads(p.read_text())["provenance"] if p.exists() else None
            return "OK" if prov == "FULL" else f"PENDING FULL RUN (thresholds={prov})"
        if name == "01_audit":
            return "OK" if MODE == "full" else "PARTIAL (game-side c/h/i PENDING FULL RUN)"
        if name == "02_combine_envelope":
            p = out_path("envelope", "_DONE")
            m = json.loads(p.read_text()).get("mode") if p.exists() else None
            return "OK (FULL)" if m == "full" else "OK (sample)"
        sub, fn = SUMMARY_FILE[name]
        p = out_path(sub, fn)
        if p.exists():
            st = str(pd.read_csv(p)["status"].iloc[0])
            return st if st == "PENDING FULL RUN" else "OK"
    except Exception:
        pass
    return "OK"

FORCE_ARGS = ["--force"] if FORCE else []
RUN_RESULTS = []
print(f"running pipeline in '{MODE}' mode ...\n")
for name in STAGE_ORDER:
    res = run_stage(name, ["--mode", MODE, *FORCE_ARGS])
    if res["ok"]:
        res["status"] = stage_output_status(name)
    else:
        res["status"] = "PENDING FULL RUN" if name in GAME_SIDE else "FAILED"
    RUN_RESULTS.append(res)
    print(f"[{res['status']:<44}] {name:<20} {res['seconds']:>7.2f}s")
    if res["error"]:
        print(f"           {res['error']}")

## 4. Synthetic-fixture demonstration (game-side path)

The Pi does not hold the full game-tracking files, so the game-side stages emit
`PENDING FULL RUN` placeholders above. To show that path **actually runs**, this cell drives
the committed, schema-compliant fixture (`fixtures/synthetic/`) with a **planted binding
ceiling** and a **null** control. Fixture outputs are labelled *not a finding* — they prove
the plumbing, nothing about the NFL.

In [ ]:
SYNTH_RESULTS = []
if RUN_SYNTHETIC_DEMO:
    for regime in SYNTHETIC_REGIMES:
        print(f"--- synthetic regime: {regime} ---")
        for name in ("00_thresholds", "03_game_demand", "04_models", "05_validate"):
            res = run_stage(name, ["--synthetic", "--synthetic-regime", regime, *FORCE_ARGS])
            res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
            SYNTH_RESULTS.append(res)
            print(f"  [{res['status']:<14}] {name:<20} {res['seconds']:>7.2f}s"
                  + ("" if res["ok"] else f"  {res['error']}"))
else:
    print("synthetic demo skipped (RUN_SYNTHETIC_DEMO=False)")

## 5. Writeup figures (≤5)

Figures are rendered from the pipeline outputs into `outputs/figures/` and shown inline.
Pre-full-run, the **Combine-side** figures (1–2) are real (the Combine file is complete);
the **game-side** figures (3–5) fall back to the synthetic fixture and carry its *not a
finding* label. When `MODE="full"` has produced real outputs, the same code picks those up
automatically and labels them `FULL`.

### Visualisation briefs

| # | chart_type | data_source | columns_required | output_filename |
|---|---|---|---|---|
| 1 | line + band | `envelope/per_player_envelope.csv` | `combine_position, env_vmax, env_tau, peak_speed` | `fig1_combine_envelope_decay.png` |
| 2 | bar (ICC / VIF) | `envelope/feature_selection.csv` | `feature, icc, vif_final, retained` | `fig2_feature_selection_icc_vif.png` |
| 3 | grouped bar | `models/*/exceedance_utilization.csv` | `pooling, exceedance_rate, u_median` | `fig3_exceedance_utilization.png` |
| 4 | line + error bars | `models/*/model_coefficients.csv` | `pooling, tau, estimate, ci_lo, ci_hi, unstable` | `fig4_quantile_slope_vs_tau.png` |
| 5 | multi-panel | `validate/*/{cv_results,measurement_error,permutation}.csv` | `tau, model, pinball, slope_uncorrected, slope_corrected, p_two_sided` | `fig5_validation_cv_permutation.png` |

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# ---- figure helpers (output readers are defined in the bootstrap cell) ------
def save_and_show(fig, filename):
    path = FIGDIR / filename
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    print(f"saved -> {path.relative_to(REPO)}   ({path.stat().st_size} bytes)")
    return path

def models_series():
    """(label, dir, is_synthetic) tuples for Phase-4 outputs: real if present, else synthetic."""
    real = out_path("models", "model_coefficients.csv")
    if real.exists():
        return [("FULL", out_path("models"), False)]
    out = []
    for reg in ("binding", "null"):
        d = out_path("models", "synthetic", reg)
        if (d / "model_coefficients.csv").exists():
            out.append((f"synthetic:{reg}", d, True))
    return out

def validate_series():
    real = out_path("validate", "cv_results.csv")
    if real.exists():
        return [("FULL", out_path("validate"), False)]
    out = []
    for reg in ("binding", "null"):
        d = out_path("validate", "synthetic", reg)
        if (d / "cv_results.csv").exists():
            out.append((f"synthetic:{reg}", d, True))
    return out

FIGURES = []

### Figure 1 — Combine ceiling envelope (`v(t)=env_vmax·(1−e^{−t/env_tau})`)

In [ ]:
env = read_csv(out_path("envelope", "per_player_envelope.csv"))
if not len(env):
    print("envelope outputs missing — run 02_combine_envelope first")
else:
    prov1 = prov_of(env)
    t = np.linspace(0.0, 3.0, 200)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    for pos, col in (("WR", "C0"), ("TE", "C1")):
        sub = env[env["combine_position"] == pos]
        vm, tau = to_num(sub["env_vmax"]).to_numpy(), to_num(sub["env_tau"]).to_numpy()
        m = np.isfinite(vm) & np.isfinite(tau) & (tau > 0)
        if not m.any():
            continue
        curves = vm[m, None] * (1.0 - np.exp(-t[None, :] / tau[m, None]))
        a1.plot(t, np.median(curves, 0), color=col, label=f"{pos} median (n={m.sum()})")
        a1.fill_between(t, np.percentile(curves, 25, 0), np.percentile(curves, 75, 0),
                        color=col, alpha=0.18)
        a2.scatter(to_num(sub["peak_speed"]), to_num(sub["env_vmax"]),
                   s=14, alpha=0.6, color=col, label=pos)
    a1.set_xlabel("time from movement onset (s)"); a1.set_ylabel("speed (yd/s)")
    a1.set_title("Fitted ceiling envelope v(t)"); a1.legend(fontsize=8)
    lim = list(a2.get_xlim()) + list(a2.get_ylim())
    lo, hi = min(lim), max(lim)
    a2.plot([lo, hi], [lo, hi], "k--", lw=1, label="y=x")
    a2.set_xlabel("model-free peak speed (yd/s)"); a2.set_ylabel("fitted ceiling env_vmax (yd/s)")
    a2.set_title("Fitted ceiling vs raw peak"); a2.legend(fontsize=8)
    fig.suptitle(f"Fig 1 — Combine ceiling envelope   [{prov1}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig1_combine_envelope_decay.png"))

### Figure 2 — Combine feature selection (ICC floor → VIF pruning)

In [ ]:
fs = read_csv(out_path("envelope", "feature_selection.csv"))
if not len(fs):
    print("feature-selection outputs missing — run 02_combine_envelope first")
else:
    prov2 = prov_of(fs)
    fs = fs.copy()
    fs["_vif"] = to_num(fs["vif_final"]).fillna(to_num(fs["vif_full_pool"]))
    fs["_ret"] = is_true(fs["retained"])
    fs = fs.sort_values("icc")
    colors = ["C2" if r else "C7" for r in fs["_ret"]]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.0), sharey=True)
    a1.barh(fs["feature"], to_num(fs["icc"]), color=colors)
    a1.axvline(0.5, ls="--", c="k", lw=1); a1.set_xlabel("test–retest ICC(1,1)")
    a1.set_title("ICC  (kept = green; floor 0.5)")
    a2.barh(fs["feature"], fs["_vif"], color=colors)
    a2.axvline(5.0, ls="--", c="k", lw=1); a2.set_xlabel("VIF (reported)")
    a2.set_title("VIF  (ceiling 5.0)")
    fig.suptitle(f"Fig 2 — Combine feature selection   [{prov2}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig2_feature_selection_icc_vif.png"))

### Figure 3 — Exceedance & utilization (game side)

In [ ]:
series = models_series()
if not series:
    print("no Phase-4 model outputs yet (run 04_models, or the synthetic demo)")
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    pools = ["pooled", "WR", "TE"]
    x = np.arange(len(pools)); w = 0.8 / max(len(series), 1)
    synth = False
    for i, (label, d, is_synth) in enumerate(series):
        synth = synth or is_synth
        ex = read_csv(Path(d) / "exceedance_utilization.csv")
        if not len(ex):
            continue
        ex = ex.set_index("pooling")
        xs = x + (i - (len(series) - 1) / 2) * w
        a1.bar(xs, [to_num(ex["exceedance_rate"]).get(p, np.nan) for p in pools],
               width=w, label=label)
        a2.bar(xs, [to_num(ex["u_median"]).get(p, np.nan) for p in pools], width=w, label=label)
    a1.axhline(0.25, ls="--", c="r", lw=1); a1.text(0.01, 0.26, "reframe >0.25",
                                                    color="r", fontsize=7, transform=a1.get_yaxis_transform())
    a1.set_xticks(x); a1.set_xticklabels(pools); a1.set_ylabel("exceedance rate")
    a1.set_title("Share of efforts beating the ceiling"); a1.legend(fontsize=8)
    a2.axhline(1.0, ls="--", c="k", lw=1); a2.set_xticks(x); a2.set_xticklabels(pools)
    a2.set_ylabel("median utilization u = effort / ceiling")
    a2.set_title("Utilization (1.0 = at ceiling)"); a2.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 3 — Exceedance & utilization   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig3_exceedance_utilization.png"))

### Figure 4 — Quantile slope vs τ (does the link strengthen toward the tail?)

In [ ]:
series = models_series()
if not series:
    print("no Phase-4 model outputs yet (run 04_models, or the synthetic demo)")
else:
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    styles = {"WR": "-o", "TE": "--s"}
    synth = False
    for label, d, is_synth in series:
        synth = synth or is_synth
        mc = read_csv(Path(d) / "model_coefficients.csv")
        if not len(mc):
            continue
        mc = mc[(mc["feature"] == "env_vmax") & (mc["controls"] == "without_draft")]
        for pool, ls in styles.items():
            sub = mc[mc["pooling"] == pool].sort_values("tau")
            if not len(sub):
                continue
            est = to_num(sub["reported"])          # blank (null) reported -> NaN -> not plotted
            lo, hi = to_num(sub["ci_lo"]), to_num(sub["ci_hi"])
            ax.errorbar(to_num(sub["tau"]), est, yerr=[est - lo, hi - est],
                        fmt=ls, capsize=3, alpha=0.9, label=f"{label} · {pool}")
            un = is_true(sub["unstable"]) & est.notna()
            ax.scatter(to_num(sub["tau"])[un], est[un], marker="x", s=45, color="k", zorder=5)
    ax.axhline(0.0, lw=1, color="k", alpha=0.5)
    ax.set_xlabel("τ (quantile)"); ax.set_ylabel("slope on ceiling env_vmax  (in-game speed per yd/s)")
    ax.set_title("Quantile slope vs τ  (×  = bootstrap-unstable; blanks = reported null)")
    ax.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 4 — Ceiling → game link across the tail   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig4_quantile_slope_vs_tau.png"))

### Figure 5 — Validation: player-grouped CV vs controls-only, measurement error, permutation

In [ ]:
series = validate_series()
if not series:
    print("no Phase-5 validation outputs yet (run 05_validate, or the synthetic demo)")
else:
    label, d, is_synth = series[0]         # binding (or real) is the reference
    cv = read_csv(Path(d) / "cv_results.csv")
    me = read_csv(Path(d) / "measurement_error.csv")
    pm = read_csv(Path(d) / "permutation.csv")
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.2))

    if len(cv):
        g = cv.groupby(["tau", "model"])["pinball"].apply(lambda s: to_num(s).mean()).unstack()
        taus = list(g.index)
        x = np.arange(len(taus)); w = 0.38
        if "full" in g:      a1.bar(x - w/2, g["full"],          width=w, label="full")
        if "controls_only" in g: a1.bar(x + w/2, g["controls_only"], width=w, label="controls-only baseline")
        a1.set_xticks(x); a1.set_xticklabels([str(t) for t in taus])
    a1.set_xlabel("τ"); a1.set_ylabel("mean pinball loss (lower = better)")
    a1.set_title("Player-grouped CV vs baseline"); a1.legend(fontsize=8)

    if len(me):
        mt = to_num(me["tau"])
        a2.plot(mt, to_num(me["slope_uncorrected"]), "-o", label="uncorrected")
        a2.plot(mt, to_num(me["slope_corrected"]), "--s", label="ICC-attenuation corrected")
        if len(me) and "reliability_icc" in me:
            a2.set_title(f"Measurement error\n(reliability ICC ≈ {to_num(me['reliability_icc']).mean():.3f})",
                         fontsize=10)
    a2.axhline(0.0, lw=1, color="k", alpha=0.5)
    a2.set_xlabel("τ"); a2.set_ylabel("slope"); a2.legend(fontsize=8)

    if len(pm):
        r = pm.iloc[0]
        obs = abs(float(to_num(pd.Series([r["observed_slope"]]).iloc[0])))
        p = float(to_num(pd.Series([r["p_two_sided"]]).iloc[0]))
        a3.bar(["observed\n|slope|"], [obs], color="C3", width=0.5)
        a3.axhline(0.0, lw=1, color="k", alpha=0.5)
        a3.set_title(f"Permutation null (τ={r['tau']})\np = {p:.3f}  "
                     f"({int(r['n_ge'])}/{int(r['n_reps'])} perms ≥ obs)", fontsize=9)
        a3.set_ylabel("|slope|")
    tag = "SYNTHETIC FIXTURE (not a finding)" if is_synth else "FULL"
    fig.suptitle(f"Fig 5 — Validation   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig5_validation_cv_permutation.png"))

## 6. Run summary

A consolidated status for the whole run, plus the `PENDING FULL RUN` items and the per-stage
wall-time / peak-RAM log (`outputs/run_log.csv`).

In [ ]:
def stage_status(res):
    return res.get("status") or ("OK" if res["ok"] else "FAILED")

summary = pd.DataFrame([{"stage": r["stage"], "status": stage_status(r),
                         "seconds": r["seconds"], "error": r["error"]} for r in RUN_RESULTS])
if SYNTH_RESULTS:
    summary = pd.concat([summary, pd.DataFrame(
        [{"stage": f"{r['stage']} [synthetic]", "status": stage_status(r),
          "seconds": r["seconds"], "error": r["error"]} for r in SYNTH_RESULTS])],
        ignore_index=True)

prov_file = out_path("thresholds.json")
prov = json.loads(prov_file.read_text())["provenance"] if prov_file.exists() else "n/a"
pending = summary.loc[summary["status"].str.contains("PENDING"), "stage"].tolist()

print(f"MODE = {MODE}   ·   thresholds provenance = {prov}")
print(f"figures written  : {len(FIGURES)} of max 5  ->  {FIGDIR.relative_to(REPO)}/")
print(f"PENDING FULL RUN : {pending if pending else 'none'}")
display(summary)

rl = out_path("run_log.csv")
if rl.exists():
    print("\nlast stage-run log rows (outputs/run_log.csv):")
    display(read_csv(rl).tail(10))

### Next steps

* **On the Pi / laptop without the full tracking files:** leave `MODE="sample"`. Stages that
  need the full game data report `PENDING FULL RUN`; the synthetic demo shows the game-side
  path runs end-to-end.
* **With the full data (laptop or Kaggle):** put `game_tracking_{2023,24,25}.csv` (and the
  Combine files) in one folder, set `DATA_DIR` to it (or attach the Kaggle dataset and let
  auto-detect find it), set `MODE="full"`, then **Run All** again. The freeze gate
  (`notes/prereg.md`, frozen) plus the audit go/no-go gate are enforced by the engine before
  any full game-side stage runs.

Engine: `src/` · Spec: `TASK.md` · Pre-registration: `notes/prereg.md` · Decisions:
`notes/decisions.md`.